# Winning Combinations

Each combo explicitly lists which mechanisms are stacked.

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Prior (single ideas)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    for n,s,sw in [('revise','st10','revise'),('JEPA','st04','jepa_w0.1'),('sparsity','st08','sparsity0.5')]:
        for t in ['cifar10','sort','mazes']:
            sub = df_prior[(df_prior.stage==s)&(df_prior.sweep==sw)&(df_prior.task==t)]
            if not sub.empty:
                m=sub.best_test_acc.mean()*100; bl=BASELINE_ACC[t]*100
                print(f'{n:10s} {t:10s}: {m:.1f}% ({m-bl:+.1f}pp)')
else: print('no prior')

## Combo Definitions

### revise + JEPA

Visual ceiling: revise refines output while JEPA regularizes latent state.
Δ = +draft_mode='revise' +draft_revise_weight=0.1 +draft_corrupt_prob=0.15 +draft_block_size=2
    +cross_tick_jepa_weight=0.1 +jepa defaults

5 seeds × ['cifar10','mazes'].

In [ ]:
exps_revise_jepa = []
for task in ['cifar10','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(5):
        exps_revise_jepa.append(Experiment(
            f'{task}_revise_jepa_s{s}', task, module,
            {**base, 'seed': s,
     'draft_mode': 'revise', 'draft_revise_weight': 0.1,
     'draft_corrupt_prob': 0.15, 'draft_block_size': 2,
     'cross_tick_jepa_weight': 0.1,
     'cross_tick_jepa_hidden_dim': 128,
     'cross_tick_jepa_predictor_depth': 2,
     'cross_tick_jepa_dropout': 0.0}))
print(f'{len(exps_revise_jepa)} runs')

### revise + sparsity

Sort ceiling: two independent mechanisms (top-k sparsity + draft-revise).
Δ = +draft_mode='revise' +revise params +topk_neurons=0.5

5 seeds × ['sort','mazes'].

In [ ]:
exps_revise_spar = []
for task in ['sort','mazes']:
    module, base = BASE_CONFIGS[task]
    for s in range(5):
        exps_revise_spar.append(Experiment(
            f'{task}_revise_spar_s{s}', task, module,
            {**base, 'seed': s,
     'draft_mode': 'revise', 'draft_revise_weight': 0.1,
     'draft_corrupt_prob': 0.15, 'draft_block_size': 2,
     'topk_neurons': 0.5}))
print(f'{len(exps_revise_spar)} runs')

### JEPA + sparsity

Representation synergy: JEPA regularizes + sparsity enforces efficient coding.
Δ = +cross_tick_jepa_weight=0.1 +jepa defaults +topk_neurons=0.5

5 seeds × ['cifar10','sort'].

In [ ]:
exps_jepa_spar = []
for task in ['cifar10','sort']:
    module, base = BASE_CONFIGS[task]
    for s in range(5):
        exps_jepa_spar.append(Experiment(
            f'{task}_jepa_spar_s{s}', task, module,
            {**base, 'seed': s,
     'cross_tick_jepa_weight': 0.1,
     'cross_tick_jepa_hidden_dim': 128,
     'cross_tick_jepa_predictor_depth': 2,
     'cross_tick_jepa_dropout': 0.0,
     'topk_neurons': 0.5}))
print(f'{len(exps_jepa_spar)} runs')

### full stack (revise + JEPA + sparsity)

All three winners combined. Does stacking help or interfere?
Δ = +revise +JEPA +sparsity (all params from above)

5 seeds × ['cifar10','sort'].

In [ ]:
exps_full_stack = []
for task in ['cifar10','sort']:
    module, base = BASE_CONFIGS[task]
    for s in range(5):
        exps_full_stack.append(Experiment(
            f'{task}_full_stack_s{s}', task, module,
            {**base, 'seed': s,
     'draft_mode': 'revise', 'draft_revise_weight': 0.1,
     'draft_corrupt_prob': 0.15, 'draft_block_size': 2,
     'cross_tick_jepa_weight': 0.1,
     'cross_tick_jepa_hidden_dim': 128,
     'cross_tick_jepa_predictor_depth': 2,
     'cross_tick_jepa_dropout': 0.0,
     'topk_neurons': 0.5}))
print(f'{len(exps_full_stack)} runs')

## Run + Analyze

In [ ]:
exps = exps_revise_jepa + exps_revise_spar + exps_jepa_spar + exps_full_stack
print(f'Total: {len(exps)} experiments')
run_all(exps, gpus=8, log_root='logs/deep/04_combos', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/04_combos')

In [ ]:
status('logs/deep/04_combos')

In [ ]:
df = collect('logs/deep/04_combos')
if not df.empty:
    df['combo'] = df.name.apply(lambda n: '+'.join([p for p in ['revise','jepa','spar'] if p in n]))
    plot_delta_bars(df, 'Combos vs baseline', 'figures/04_delta.png')
    print(significance_test(df).to_string(index=False))
    print(summary_stats(df, groupby=('combo','task')))
else: print('No results yet.')